# CS6886W – Assignment 1: VGG6 on CIFAR-10
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cs25m506/vgg6-cifar10-assignment/blob/master/notebooks/CS6886W_A1.ipynb)

> **Runtime:** GPU required — *Runtime → Change runtime type → T4 GPU*

This notebook drives all Q1/Q2/Q3 experiments end-to-end.  
Metrics are logged to **Weights & Biases** automatically every epoch.


## 0 · Install Dependencies

In [ ]:
!pip install -q torch torchvision wandb tqdm numpy
import torch, torchvision, wandb
print(f"torch       {torch.__version__}")
print(f"torchvision {torchvision.__version__}")
print(f"wandb       {wandb.__version__}")
print(f"CUDA        {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1 · Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/cs25m506/vgg6-cifar10-assignment.git"  # ← update
REPO_DIR = "/content/vgg6-cifar10-assignment"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned – pulling latest…")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 2 · Mount Google Drive (persist checkpoints across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/CS6886W_A1"
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
print(f"Checkpoints will persist at {DRIVE_DIR}/checkpoints/")

# Symlink so train.py's default ./checkpoints writes to Drive
!ln -sfn {DRIVE_DIR}/checkpoints /content/vgg6-cifar10-assignment/checkpoints
print("Symlink created: ./checkpoints → Drive")

## 3 · W&B Login

In [ ]:
# ── Option A: Colab Secrets (recommended — no key exposed in notebook) ──────
# 1. Click the 🔑 key icon in the left sidebar
# 2. Add a secret named  WANDB_API_KEY  with your key from wandb.ai/authorize
# 3. Toggle "Notebook access" ON, then run this cell

import os
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    print("Colab Secrets not available — falling back to interactive login")

import wandb
wandb.login()   # no-op if WANDB_API_KEY env var is already set

# ── Set your W&B username / team here (used in Q3 fetch) ─────────────────
WANDB_ENTITY  = "<balaji-kulkarni92-org>"   # ← replace once, used everywhere below
WANDB_PROJECT = "CS6886W-A1-VGG6-CIFAR10"
print(f"Entity : {WANDB_ENTITY}")
print(f"Project: {WANDB_PROJECT}")

## Q1 · Training Baseline (10 pts)
**(a)** Augmentation: `RandomHorizontalFlip + RandomCrop(32, pad=4) + Normalize`  
**(b)** Best config: **ReLU · Adam · lr=1e-3 · bs=128 · 30 epochs**  
**(c)** Loss/accuracy curves auto-logged to W&B


In [ ]:
!python train.py \
    --activation   relu \
    --optimizer    adam \
    --lr           1e-3 \
    --batch_size   128  \
    --num_epochs   30   \
    --seed         42   \
    --wandb_entity {balaji-kulkarni92-org}   \
    --wandb_project {WANDB_PROJECT} \
    --run_name     "Q1_baseline_relu_adam"

## Q2(a) · Activation Function Sweep (5 runs, ~55 min)
Keeps all other hyper-params fixed; varies activation: **ReLU · Sigmoid · Tanh · SiLU · GELU**


In [ ]:
activations = ["relu", "sigmoid", "tanh", "silu", "gelu"]

for act in activations:
    print(f"\n{'='*55}")
    print(f"  Q2a — activation = {act}")
    print('='*55)
    !python train.py \
        --activation   {act}            \
        --optimizer    adam             \
        --lr           1e-3             \
        --batch_size   128              \
        --num_epochs   30               \
        --seed         42               \
        --wandb_entity {WANDB_ENTITY}   \
        --wandb_project {WANDB_PROJECT} \
        --run_name     "Q2a_act={act}"

## Q2(b) · Optimizer Sweep (6 runs, ~65 min)
SGD-family: `lr=0.01`  |  Adaptive: `lr=1e-3`


In [ ]:
optimizer_lrs = {
    "sgd":          0.01,
    "nesterov_sgd": 0.01,
    "adam":         1e-3,
    "adagrad":      1e-2,
    "rmsprop":      1e-3,
    "nadam":        1e-3,
}

for opt, lr in optimizer_lrs.items():
    print(f"\n{'='*55}")
    print(f"  Q2b — optimizer = {opt},  lr = {lr}")
    print('='*55)
    !python train.py \
        --activation   relu             \
        --optimizer    {opt}            \
        --lr           {lr}             \
        --batch_size   128              \
        --num_epochs   30               \
        --seed         42               \
        --wandb_entity {WANDB_ENTITY}   \
        --wandb_project {WANDB_PROJECT} \
        --run_name     "Q2b_opt={opt}"

## Q2(c) · Batch Size / LR / Epoch Sweep (12 runs, ~130 min)
Run the three sub-grids sequentially.  
💡 *If Colab disconnects, just re-run from the next configuration — completed runs are already on W&B.*


In [ ]:
# ── Batch size sweep (fix lr=1e-3, epochs=30) ──────────────────────────────
for bs in [32, 64, 128, 256]:
    print(f"\n  Q2c — batch_size = {bs}")
    !python train.py \
        --activation   relu             \
        --optimizer    adam             \
        --lr           1e-3             \
        --batch_size   {bs}             \
        --num_epochs   30               \
        --seed         42               \
        --wandb_entity {WANDB_ENTITY}   \
        --wandb_project {WANDB_PROJECT} \
        --run_name     "Q2c_bs={bs}"

In [ ]:
# ── LR sweep (fix bs=128, epochs=30) ────────────────────────────────────────
for lr in ["1e-4", "5e-4", "1e-3", "5e-3"]:
    print(f"\n  Q2c — lr = {lr}")
    !python train.py \
        --activation   relu             \
        --optimizer    adam             \
        --lr           {lr}             \
        --batch_size   128              \
        --num_epochs   30               \
        --seed         42               \
        --wandb_entity {WANDB_ENTITY}   \
        --wandb_project {WANDB_PROJECT} \
        --run_name     "Q2c_lr={lr}"

In [ ]:
# ── Epoch sweep (fix bs=128, lr=1e-3) ───────────────────────────────────────
for ep in [10, 20, 30, 50]:
    print(f"\n  Q2c — epochs = {ep}")
    !python train.py \
        --activation   relu             \
        --optimizer    adam             \
        --lr           1e-3             \
        --batch_size   128              \
        --num_epochs   {ep}             \
        --seed         42               \
        --wandb_entity {WANDB_ENTITY}   \
        --wandb_project {WANDB_PROJECT} \
        --run_name     "Q2c_ep={ep}"

## Q3 · Plots & Analysis (10 pts)
Pulls all run histories from W&B and plots them inline.  
The **Parallel Coordinates** plot is generated directly from W&B UI (see README instructions).


In [ ]:
import wandb
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Fetch all runs ────────────────────────────────────────────────────────────
api  = wandb.Api()
runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")
print(f"Found {len(runs)} runs in {WANDB_ENTITY}/{WANDB_PROJECT}")

histories  = {}
summaries  = []

for run in runs:
    hist = run.history(keys=["epoch", "train/loss", "train/acc", "val/loss", "val/acc"])
    if not hist.empty:
        hist = hist.sort_values("epoch").reset_index(drop=True)
        histories[run.name] = hist
    summaries.append({
        "run_name":     run.name,
        "activation":   run.config.get("activation", "—"),
        "optimizer":    run.config.get("optimizer",  "—"),
        "lr":           run.config.get("lr",         "—"),
        "batch_size":   run.config.get("batch_size", "—"),
        "num_epochs":   run.config.get("num_epochs", "—"),
        "best_val_acc": run.summary.get("best_val_acc", None),
    })

df_summary = pd.DataFrame(summaries).sort_values("best_val_acc", ascending=False)
print("\nBest Val Acc Summary:")
print(df_summary.to_string(index=False))

In [ ]:
# ── Plot 1: Train Loss ────────────────────────────────────────────────────────
fig = go.Figure()
for name, hist in histories.items():
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["train/loss"],
                             mode="lines", name=name))
fig.update_layout(title="Training Loss vs Epoch",
                  xaxis_title="Epoch", yaxis_title="Train Loss", height=450)
fig.show()

In [ ]:
# ── Plot 2: Validation Loss ───────────────────────────────────────────────────
fig = go.Figure()
for name, hist in histories.items():
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["val/loss"],
                             mode="lines", name=name))
fig.update_layout(title="Validation Loss vs Epoch",
                  xaxis_title="Epoch", yaxis_title="Val Loss", height=450)
fig.show()

In [ ]:
# ── Plot 3: Training Accuracy ─────────────────────────────────────────────────
fig = go.Figure()
for name, hist in histories.items():
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["train/acc"],
                             mode="lines", name=name))
fig.update_layout(title="Training Accuracy vs Epoch",
                  xaxis_title="Epoch", yaxis_title="Train Acc (%)", height=450)
fig.show()

In [ ]:
# ── Plot 4: Validation Accuracy ───────────────────────────────────────────────
fig = go.Figure()
for name, hist in histories.items():
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["val/acc"],
                             mode="lines", name=name))
fig.update_layout(title="Validation Accuracy vs Epoch",
                  xaxis_title="Epoch", yaxis_title="Val Acc (%)", height=450)
fig.show()

In [ ]:
# ── Plot 5: Val Accuracy Scatter (vs Step) ───────────────────────────────────
fig = go.Figure()
for name, hist in histories.items():
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["val/acc"],
                             mode="markers", name=name,
                             marker=dict(size=6)))
fig.update_layout(title="Val Accuracy Scatter vs Step",
                  xaxis_title="Step (Epoch)", yaxis_title="Val Acc (%)", height=450)
fig.show()

In [ ]:
# ── W&B Parallel Coordinates (direct link) ───────────────────────────────────
# The Parallel Coordinates plot is best generated in the W&B UI.
# Run this cell to open your project directly:
project_url = f"https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}"
print(f"Your W&B project: {project_url}")
print()
print("Steps to generate Parallel Coordinates in W&B:")
print("  1. Open the URL above")
print("  2. Go to Charts tab → click '+ Add chart' → 'Parallel coordinates'")
print("  3. Add axes: activation, optimizer, lr, batch_size, num_epochs, best_val_acc")
print("  4. Set color axis = best_val_acc")
print("  5. Download → PNG/SVG for submission")

## Q4 · Evaluate Best Checkpoint

In [ ]:
BEST_CKPT = "checkpoints/Q1_baseline_relu_adam_best.pth"   # ← update path
!python evaluate.py --checkpoint {BEST_CKPT}